## 1. Environment Setup and Library Imports

In [1]:
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# 1. Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model

# -------------------------------------------------------------------------
# 2. Hardware and Training Hyperparameters (HC Thickness Prediction - Solo RP)
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Target material & single-sensor modality
target_material = "HC"
sensor_modality = "RP"
INPUT_CHANNELS = 1

# Classification setup: 5 target sheet thicknesses
thickness_list = ["T185", "T188", "T191", "T194", "T197"]
thickness_to_id = {t: idx for idx, t in enumerate(thickness_list)}
NUM_CLASSES_THICKNESS = len(thickness_list)  # 5 classes
NUM_EPOCHS = 35

# Sample sizes to evaluate sample efficiency (stratified across the 5 classes)
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{s}" for s in sample_sizes]

# Random seeds (10 seeds for statistical significance)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# 3. Path Verification for HC Material across all 5 Thicknesses (RP Only)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 1: Verifying HC Paths for Thickness Prediction (Solo RP)...")
print("==================================================================")

all_paths_ok = True
hc_rp_paths = {}

for t in thickness_list:
    rp_p = DATA_DIR / target_material / t / sensor_modality
    exists_rp = rp_p.exists()
    
    if not exists_rp:
        all_paths_ok = False
        
    print(f"[{target_material} {t} {sensor_modality}] {rp_p} -> {'FOUND' if exists_rp else 'MISSING'}")
    hc_rp_paths[t] = str(rp_p)

print(f"\nPath status: {'ALL PATHS OK' if all_paths_ok else 'WARNING: SOME PATHS ARE MISSING'}")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Target Material: {target_material}")
print(f"Modality: {sensor_modality} (Input channels: {INPUT_CHANNELS})")
print(f"Thickness Classes: {thickness_to_id} (Total: {NUM_CLASSES_THICKNESS})")
print(f"Evaluation Subsets: {subset_names}")
print(f"Total Random Seeds: {len(seeds)}")

STEP 1: Verifying HC Paths for Thickness Prediction (Solo RP)...
[HC T185 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T185\RP -> FOUND
[HC T188 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T188\RP -> FOUND
[HC T191 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T191\RP -> FOUND
[HC T194 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T194\RP -> FOUND
[HC T197 RP] c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP -> FOUND

Path status: ALL PATHS OK
Device: cuda
Project root: c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data
Target Material: HC
Modality: RP (Input channels: 1)
Thickness Clas

## 2. Data Ingestion & In-Memory Preloading (RP Force Sensor)

In [2]:
import os
import torch
from torch.utils.data import DataLoader
from src.pkldataset import PKLDataset

# -------------------------------------------------------------------------
# STEP 2: Load HC Data Across All 5 Thicknesses (RP Only) & Assign Thickness Labels
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 2: Loading RP Force Signals for All 5 HC Sheet Thicknesses...")
print("==================================================================")

cpu_cores = os.cpu_count() or 4
num_workers = min(8, cpu_cores)

# Container by thickness: {'inputs': tensor[N, 1, L], 'targets': tensor[N]}
hc_rp_tensors = {}

for t_name in thickness_list:
    folder_path = hc_rp_paths[t_name]
    t_id = thickness_to_id[t_name]  # 0: T185, 1: T188, 2: T191, 3: T194, 4: T197
    
    # Load raw single-modality force signals (RP)
    ds = PKLDataset(folder_path=folder_path, modality="single")
    loader = DataLoader(
        ds, 
        batch_size=512, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=False,
        persistent_workers=False
    )
    
    xs = []
    for x, _ in loader:  # Discard the original 10-class wear label
        xs.append(x)
        
    inputs = torch.cat(xs, dim=0)  # Tensor shape: [N, 1, L]
    
    # Assign target label corresponding to current sheet thickness ID
    targets = torch.full((inputs.shape[0],), fill_value=t_id, dtype=torch.long)
    
    hc_rp_tensors[t_name] = {
        "inputs": inputs,
        "targets": targets
    }
    print(f"[{t_name}] ID: {t_id} | Cached samples: {inputs.shape[0]} | Tensor shape: {tuple(inputs.shape)}")

print("\nAll HC thickness datasets (RP) are successfully loaded and cached in RAM.")

STEP 2: Loading RP Force Signals for All 5 HC Sheet Thicknesses...
[T185] ID: 0 | Cached samples: 5010 | Tensor shape: (5010, 1, 2800)
[T188] ID: 1 | Cached samples: 5010 | Tensor shape: (5010, 1, 2800)
[T191] ID: 2 | Cached samples: 5010 | Tensor shape: (5010, 1, 2800)
[T194] ID: 3 | Cached samples: 5009 | Tensor shape: (5009, 1, 2800)
[T197] ID: 4 | Cached samples: 5010 | Tensor shape: (5010, 1, 2800)

All HC thickness datasets (RP) are successfully loaded and cached in RAM.


## 3. Train/Test Partitioning (RP Force Sensor)

In [3]:
import numpy as np
import torch
from torch.utils.data import TensorDataset

# -------------------------------------------------------------------------
# STEP 3: Train/Test Partitioning & Stratified Sampling Setup (HC - Solo RP)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 3: Partitioning Data into Train Pools and Consolidated Test Set...")
print("==================================================================")

TEST_RATIO = 0.20
EVAL_SEED = 42

np.random.seed(EVAL_SEED)

train_pools = {}
test_inputs_list, test_targets_list = [], []

for t_name in thickness_list:
    inputs = hc_rp_tensors[t_name]["inputs"]
    targets = hc_rp_tensors[t_name]["targets"]
    n_samples = inputs.shape[0]
    
    indices = np.random.permutation(n_samples)
    n_test = int(n_samples * TEST_RATIO)
    
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    
    # Pool available to draw stratified training subsets (train_20 ... train_500)
    train_pools[t_name] = {
        "inputs": inputs[train_idx],
        "targets": targets[train_idx]
    }
    
    test_inputs_list.append(inputs[test_idx])
    test_targets_list.append(targets[test_idx])
    
    print(f"[{t_name}] Train pool: {len(train_idx)} samples | Test set: {len(test_idx)} samples")

# Concatenate all test partitions into a single evaluation tensor on device
X_test = torch.cat(test_inputs_list, dim=0).to(device)
Y_test = torch.cat(test_targets_list, dim=0).to(device)

print(f"\nConsolidated Test Set shape: {tuple(X_test.shape)}")
print(f"Consolidated Test Targets shape: {tuple(Y_test.shape)}")

# -------------------------------------------------------------------------
# Stratified Sampling Helper for Multi-Class Thickness Prediction (5 Classes)
# -------------------------------------------------------------------------
def get_balanced_thickness_subset(train_pools, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset across all 5 thickness classes.
    For example: 20 samples -> 4 samples per thickness class (T185..T197).
    Returns concatenated input and target tensors for the training batch.
    """
    np.random.seed(seed)
    num_classes = len(thickness_list)  # 5 classes
    samples_per_class = total_samples // num_classes
    
    selected_x, selected_y = [], []
    
    for t_name in thickness_list:
        pool_x = train_pools[t_name]["inputs"]
        pool_y = train_pools[t_name]["targets"]
        total_available = pool_x.shape[0]
        
        if total_available < samples_per_class:
            raise ValueError(
                f"Not enough samples in {t_name} pool: "
                f"{total_available} available vs {samples_per_class} requested."
            )
            
        chosen_indices = np.random.choice(total_available, size=samples_per_class, replace=False)
        selected_x.append(pool_x[chosen_indices])
        selected_y.append(pool_y[chosen_indices])
        
    batch_x = torch.cat(selected_x, dim=0)
    batch_y = torch.cat(selected_y, dim=0)
    
    # Shuffle the concatenated balanced batch
    perm = torch.randperm(batch_x.size(0))
    return batch_x[perm], batch_y[perm]

# -------------------------------------------------------------------------
# Results Container Initialization
# -------------------------------------------------------------------------
results = {
    s_name: {
        "overall": [],
        **{t: [] for t in thickness_list}
    }
    for s_name in subset_names
}

print(f"\nStep 3 completed successfully:")
print(f"  Target Classes: {thickness_list} ({NUM_CLASSES_THICKNESS} classes)")
print(f"  Evaluation Subsets: {subset_names}")
print(f"  Random Seeds: {len(seeds)}")

STEP 3: Partitioning Data into Train Pools and Consolidated Test Set...
[T185] Train pool: 4008 samples | Test set: 1002 samples
[T188] Train pool: 4008 samples | Test set: 1002 samples
[T191] Train pool: 4008 samples | Test set: 1002 samples
[T194] Train pool: 4008 samples | Test set: 1001 samples
[T197] Train pool: 4008 samples | Test set: 1002 samples

Consolidated Test Set shape: (5009, 1, 2800)
Consolidated Test Targets shape: (5009,)

Step 3 completed successfully:
  Target Classes: ['T185', 'T188', 'T191', 'T194', 'T197'] (5 classes)
  Evaluation Subsets: ['train_20', 'train_50', 'train_100', 'train_200', 'train_300', 'train_400', 'train_500']
  Random Seeds: 10


## 4. Training and Evaluation - RP

In [4]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -------------------------------------------------------------------------
# 1. Fast in-memory evaluation helper for thickness prediction
# -------------------------------------------------------------------------
def fast_eval_thickness(model, x_test, y_test, batch_size=512):
    model.eval()
    all_preds = []
    total = y_test.size(0)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = x_test[i:i + batch_size]
            preds = model(x).argmax(dim=1)
            all_preds.append(preds)
            
    all_preds = torch.cat(all_preds, dim=0)
    
    # Overall accuracy across all 5 thickness classes
    overall_acc = 100.0 * (all_preds == y_test).sum().item() / total
    
    # Accuracy broken down by each thickness class
    class_accs = {}
    for t_name, t_id in thickness_to_id.items():
        mask = (y_test == t_id)
        if mask.sum().item() > 0:
            acc = 100.0 * (all_preds[mask] == t_id).sum().item() / mask.sum().item()
            class_accs[t_name] = acc
        else:
            class_accs[t_name] = 0.0
            
    return overall_acc, class_accs

# -------------------------------------------------------------------------
# 2. Reset results container to clear previous runs
# -------------------------------------------------------------------------
results = {
    s_name: {
        "overall": [],
        **{t: [] for t in thickness_list}
    }
    for s_name in subset_names
}

# -------------------------------------------------------------------------
# 3. Stabilized Training Loop for HC Thickness Prediction (Solo RP)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 4: Training 1D-CNN (RP Only) for HC Thickness Prediction (5 Classes)...")
print("==================================================================")

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running HC Thickness Prediction (RP) for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)
    
    for s_size, s_name in zip(sample_sizes, subset_names):
        # Stratified extraction from HC pools (equal samples per thickness)
        train_x, train_y = get_balanced_thickness_subset(
            train_pools=train_pools,
            total_samples=s_size,
            seed=seed
        )
        
        # Adaptive batch sizing: 32 for small sets, 64 for >= 200 samples
        if s_size < 64:
            curr_batch_size = max(8, s_size // NUM_CLASSES_THICKNESS)  # At least 1-2 per class
        elif s_size < 200:
            curr_batch_size = 32
        else:
            curr_batch_size = 64
            
        train_loader = DataLoader(
            TensorDataset(train_x, train_y),
            batch_size=curr_batch_size,
            shuffle=True
        )
        
        # 1D-CNN: 1 input channel (RP), 5 output classes (T185..T197)
        model = get_model(input_channels=INPUT_CHANNELS, num_classes=NUM_CLASSES_THICKNESS).to(device)
        
        # Stabilized optimization parameters
        optimizer = optim.Adam(model.parameters(), lr=7e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.2)
        
        # Train from scratch
        model = train_model(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=NUM_EPOCHS,
            device=device
        )
        
        # Evaluate on the consolidated unseen test set
        overall_acc, class_accs = fast_eval_thickness(
            model=model,
            x_test=X_test,
            y_test=Y_test,
            batch_size=512
        )
        
        # Store metrics
        results[s_name]["overall"].append(overall_acc)
        for t_name in thickness_list:
            results[s_name][t_name].append(class_accs[t_name])
            
        print(f"[Seed {seed}] {s_name} ({s_size} samples) -> Test Overall Acc: {overall_acc:.2f}% | " +
              " | ".join([f"{t}: {class_accs[t]:.1f}%" for t in thickness_list]))

print("\nStep 4 completed successfully. Results stored in 'results'.")

STEP 4: Training 1D-CNN (RP Only) for HC Thickness Prediction (5 Classes)...

>>> Running HC Thickness Prediction (RP) for Seed: 101
Epoch 1/35, Loss: 2.1767
Epoch 2/35, Loss: 2.7331
Epoch 3/35, Loss: 1.9435
Epoch 4/35, Loss: 1.6956
Epoch 5/35, Loss: 1.9445
Epoch 6/35, Loss: 1.5836
Epoch 7/35, Loss: 1.4521
Epoch 8/35, Loss: 1.1823
Epoch 9/35, Loss: 1.4279
Epoch 10/35, Loss: 1.1708
Epoch 11/35, Loss: 1.1592
Epoch 12/35, Loss: 0.8178
Epoch 13/35, Loss: 0.9157
Epoch 14/35, Loss: 0.9839
Epoch 15/35, Loss: 0.8576
Epoch 16/35, Loss: 0.8767
Epoch 17/35, Loss: 1.0159
Epoch 18/35, Loss: 0.6370
Epoch 19/35, Loss: 0.6718
Epoch 20/35, Loss: 0.6410
Epoch 21/35, Loss: 0.5934
Epoch 22/35, Loss: 0.4843
Epoch 23/35, Loss: 0.7169
Epoch 24/35, Loss: 0.6596
Epoch 25/35, Loss: 0.5877
Epoch 26/35, Loss: 0.6004
Epoch 27/35, Loss: 0.4742
Epoch 28/35, Loss: 0.4325
Epoch 29/35, Loss: 0.6434
Epoch 30/35, Loss: 0.3954
Epoch 31/35, Loss: 0.4069
Epoch 32/35, Loss: 0.5647
Epoch 33/35, Loss: 0.5247
Epoch 34/35, Loss:

## 5. Summary of Results - RP

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------------
# STEP 5: Compile Final Summary Table and Export (HC Thickness - Solo RP)
# -------------------------------------------------------------------------
print("==================================================================")
print("STEP 5: Compiling Consolidated Results Table (HC Thickness - Solo RP)...")
print("==================================================================")

summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    
    # Per-class accuracy (recall) for each sheet thickness (sin formato de %)
    for t_name in thickness_list:
        class_scores = results[s_name][t_name]
        mean_c = np.mean(class_scores)
        std_c = np.std(class_scores)
        row[t_name] = f"{mean_c:.2f} ± {std_c:.2f}"
        
    summary_data.append(row)

# Strict column ordering: Subset -> T185 ... T197
column_order = ["Subset"] + thickness_list
df_thickness_rp = pd.DataFrame(summary_data)[column_order]

print(f"\nConsolidated Thickness Prediction Results (HC - Solo RP) [Mean ± Std over {len(seeds)} seeds]:")
display(df_thickness_rp)

# Save to CSV
output_csv = "RP_thickness_pred_HC_summary.csv"
df_thickness_rp.to_csv(output_csv, index=False)
print(f"\nFinal results successfully exported to: {output_csv}")

STEP 5: Compiling Consolidated Results Table (HC Thickness - Solo RP)...

Consolidated Thickness Prediction Results (HC - Solo RP) [Mean ± Std over 10 seeds]:


,Subset,T185,T188,T191,T194,T197
0,train_20,68.13 ± 17.84,60.76 ± 12.78,60.36 ± 15.11,72.98 ± 20.28,75.53 ± 17.81
1,train_50,83.13 ± 28.37,84.60 ± 8.34,77.76 ± 26.88,78.14 ± 26.65,85.16 ± 30.57
2,train_100,91.78 ± 5.73,88.54 ± 2.85,90.21 ± 2.22,91.54 ± 3.03,98.92 ± 1.01
3,train_200,92.92 ± 3.07,91.46 ± 2.92,91.73 ± 2.54,94.55 ± 1.87,99.26 ± 0.82
4,train_300,94.51 ± 2.09,92.50 ± 1.57,93.84 ± 1.44,96.00 ± 1.88,99.47 ± 0.54
5,train_400,94.95 ± 1.97,93.53 ± 1.86,94.18 ± 1.59,96.68 ± 1.91,99.52 ± 0.40
6,train_500,95.69 ± 1.93,93.96 ± 1.97,94.33 ± 1.89,97.08 ± 1.55,99.45 ± 0.44



Final results successfully exported to: RP_thickness_pred_HC_summary.csv
